# PMJAY PS-2 Starter Notebook (Learn + Build)
You are doing the right thing: first make a small working pipeline on one sample, then scale.

Run this notebook from top to bottom.

## Notebook Roadmap (Single-File Workflow)
Use only this notebook from top to bottom. No separate .py script is required.

What each cell does:
1. Cell 1: sets project paths and creates outputs folder.
2. Cell 2: inventories dataset files by extension.
3. Cell 3: picks one sample PDF and image for sanity checks.
4. Cell 4: defines OCR utility (direct PDF text + OCR fallback).
5. Cell 5A-5C: runs offline checks and validates OCR on one sample.
6. Cell 6: batch OCR for all PDFs (safe by default with RUN_BATCH_OCR=False).
7. Cell 7: image analysis for all images and exports image_analysis_results.csv.
8. Cell 8: aggregates OCR + image signals per claim into claims_aggregated.csv.
9. Cell 9: calls NHA vision model on images and saves model outputs.

Important safety note:
- Keep RUN_BATCH_OCR=False in Cell 6 unless you explicitly want to regenerate OCR outputs.
- Do not hardcode API secrets in notebook cells; load them from environment or local credentials file.

## Cell 1: Project paths and output folder
What this teaches: how to make code independent of hard-coded paths using `pathlib`.
Why needed: every next step depends on finding your `dataset/Claims` folder safely.

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
CLAIMS_DIR = ROOT / "dataset" / "Claims"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace root:", ROOT)
print("Claims dir exists:", CLAIMS_DIR.exists())
print("Outputs dir:", OUT_DIR)

## Cell 2: Dataset inventory
What this teaches: unstructured data projects start with inventory, not model training.
Concept: count file extensions to quickly understand data composition (PDF-heavy, image-heavy, etc.).

In [ ]:
all_files = [p for p in CLAIMS_DIR.rglob('*') if p.is_file()]
ext_counts = Counter(p.suffix.lower() for p in all_files)

print("Total files:", len(all_files))
print("Top extensions:")
for ext, cnt in ext_counts.most_common(15):
    print(f"  {ext or '[no extension]'}: {cnt}")

## Cell 3: Choose one sample PDF and one sample image
What this teaches: one-sample-first strategy.
Why needed: if one sample cannot run cleanly, batch processing will fail at scale.

In [ ]:
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

pdf_files = sorted(CLAIMS_DIR.rglob('*.pdf'))
img_files = sorted([
    p for p in CLAIMS_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in img_exts
])

sample_pdf = pdf_files[0] if pdf_files else None
sample_img = img_files[0] if img_files else None

print("PDF files found:", len(pdf_files))
print("Image files found:", len(img_files))
print("Sample PDF:", sample_pdf)
print("Sample Image:", sample_img)

## Cell 4: OCR utility (offline-safe design)
What this teaches:
- `pypdf` extracts embedded text directly (fast, no OCR needed).
- If little text is found, fallback to image-based OCR using local EasyOCR models only.

Offline rules in this notebook:
- No network calls.
- No runtime package installs.
- No model downloads at runtime (`download_enabled=False`).

In [ ]:
import os
import numpy as np
from pathlib import Path

def _get_local_easyocr_reader(project_root, languages=None, gpu=False):
    import easyocr

    langs = languages or ["en"]
    model_dir = Path(project_root) / "assets" / "easyocr_models"
    user_net_dir = Path(project_root) / "assets" / "easyocr_user_network"
    model_dir.mkdir(parents=True, exist_ok=True)
    user_net_dir.mkdir(parents=True, exist_ok=True)

    # Force EasyOCR to use local model directory only.
    os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
    os.environ["MODULE_PATH"] = str(model_dir)

    return easyocr.Reader(
        langs,
        gpu=gpu,
        model_storage_directory=str(model_dir),
        user_network_directory=str(user_net_dir),
        download_enabled=False,
    )

def _ocr_image_with_easyocr(pil_image, reader):
    arr = np.array(pil_image)
    results = reader.readtext(arr, detail=0, paragraph=True)
    return "\n".join([r for r in results if isinstance(r, str)]).strip()

def extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False, debug=False):
    text = ""
    used_method = "none"

    # Stage 1: direct text extraction (always available path)
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        chunks = []
        for page in reader.pages[:max_pages]:
            chunks.append((page.extract_text() or "").strip())
        text = "\n".join(chunks).strip()
        used_method = "pypdf"
    except Exception as e:
        if debug:
            print("Direct text extraction failed:", repr(e))

    # Stage 2: local-only OCR fallback if direct text is too short
    if len(text) < 120:
        try:
            from pdf2image import convert_from_path

            images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
            if debug:
                print("Pages converted to images:", len(images))

            ocr_reader = _get_local_easyocr_reader(ROOT, languages=["en"], gpu=use_gpu)
            ocr_chunks = [_ocr_image_with_easyocr(img, ocr_reader) for img in images]
            ocr_text = "\n".join([c for c in ocr_chunks if c]).strip()

            if len(ocr_text) > len(text):
                text = ocr_text
                used_method = "pdf2image+easyocr(local-only)"
        except Exception as e:
            if debug:
                print("OCR fallback failed:", repr(e))

    return text, used_method

## Cell 5: Run OCR on one sample and inspect output
What this teaches: validation mindset.
You save OCR text to a file and print preview so you can judge extraction quality immediately.

In [ ]:
# Cell 5A: Offline preflight (no installs, no downloads)
from pathlib import Path
import importlib
import shutil

print("=== Offline Preflight ===")
required = ["pypdf", "pdf2image", "easyocr", "PIL", "cv2", "numpy", "torch"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"OK: {pkg}")
    except Exception as e:
        print(f"MISSING: {pkg} ({e})")
        missing.append(pkg)

print("\n=== Local Assets Check ===")
model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
poppler_on_path = shutil.which("pdfinfo")

print("easyocr model dir:", model_dir, "exists:", model_dir.exists())
print("easyocr user network dir:", user_net_dir, "exists:", user_net_dir.exists())
print("pdfinfo on PATH:", poppler_on_path if poppler_on_path else "NOT FOUND")

if missing:
    print("\nRESULT: FAIL (missing python packages)")
elif not poppler_on_path:
    print("\nRESULT: FAIL (Poppler not on PATH)")
else:
    print("\nRESULT: PASS (offline prerequisites available)")

print("\nNote: This notebook does NOT install anything automatically.")

In [ ]:
# Cell 5B: Local-only OCR readiness diagnostics
import os
from pathlib import Path

model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
model_dir.mkdir(parents=True, exist_ok=True)
user_net_dir.mkdir(parents=True, exist_ok=True)

os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
os.environ["MODULE_PATH"] = str(model_dir)

all_files = sorted([p.name for p in model_dir.glob("*") if p.is_file()])
required_detector = "craft_mlt_25k.pth"
possible_recognizers = ["english_g2.pth", "latin_g2.pth"]

print("EASYOCR_MODULE_PATH:", os.environ.get("EASYOCR_MODULE_PATH"))
print("MODULE_PATH:", os.environ.get("MODULE_PATH"))
print("Model directory:", model_dir)
print("User network directory:", user_net_dir)
print("Directory file count (models):", len(all_files))
print("Model files:", all_files)

has_detector = required_detector in all_files
has_recognizer = any(x in all_files for x in possible_recognizers)

print("Detector present (craft_mlt_25k.pth):", has_detector)
print("Recognizer present (english_g2/latin_g2):", has_recognizer)

if len(all_files) == 0:
    print("WARNING: easyocr model files are not present locally yet.")
elif not has_detector or not has_recognizer:
    print("WARNING: model folder has files, but required EasyOCR weights may be missing or misnamed.")
else:
    print("OK: local model files look valid for English OCR.")

In [ ]:
# Cell 5C: Run OCR on one sample PDF and save output (offline-safe)
if sample_pdf is None:
    print("No PDF found. Check dataset path or extension cases.")
else:
    text, method = extract_text_from_pdf(sample_pdf, max_pages=2, use_gpu=False, debug=True)
    out_txt = OUT_DIR / "sample_ocr_text_easyocr.txt"
    out_txt.write_text(text, encoding="utf-8", errors="ignore")

    print("OCR method used:", method)
    print("Characters extracted:", len(text))
    print("Saved to:", out_txt)
    print("--- OCR preview (first 1200 chars) ---")
    print(text[:1200] if text else "[No text extracted]")

    if len(text) == 0:
        print("\nALERT: No text extracted.")
        print("Possible reasons:")
        print("  1. PDF has no extractable text and OCR fallback prerequisites are missing")
        print("  2. Poppler (pdfinfo) is not available")
        print("  3. EasyOCR local model files are missing, wrong, or misnamed")
        print("\nRun Cell 5A and Cell 5B, then check printed debug errors above.")

## Cell 6: Batch OCR pipeline (entire Claims batch)
What this does:
- Runs OCR for all PDFs under Claims recursively.
- Saves extracted text for each PDF.
- Creates a CSV summary for downstream adjudication.

Output files:
- outputs/ocr_batch_results.csv
- outputs/ocr_texts/*.txt

In [2]:
# Cell 6 code: Batch OCR on entire Claims folder and export CSV
import csv
import hashlib
from pathlib import Path

LOW_QUALITY_THRESHOLD = 200
MAX_PAGES = 3
RUN_BATCH_OCR = False  # Set to True only when you intentionally want to rerun batch OCR

# Self-healing setup so this cell can run independently.
if "ROOT" not in globals():
    ROOT = Path.cwd()
if "CLAIMS_DIR" not in globals():
    CLAIMS_DIR = ROOT / "dataset" / "Claims"
if "OUT_DIR" not in globals():
    OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RUN_BATCH_OCR:
    print("Batch OCR is skipped (RUN_BATCH_OCR=False).")
    print("Set RUN_BATCH_OCR=True only if you want to regenerate OCR outputs.")
else:
    pdf_files = sorted(CLAIMS_DIR.rglob("*.pdf"))
    if not pdf_files:
        raise RuntimeError(f"No PDF files found under: {CLAIMS_DIR}")

    if "extract_text_from_pdf" not in globals():
        raise RuntimeError("extract_text_from_pdf is not defined. Run Cell 4 first, then run this cell.")

    ocr_text_dir = OUT_DIR / "ocr_texts"
    ocr_text_dir.mkdir(parents=True, exist_ok=True)

    records = []

    print("=" * 72)
    print("FULL BATCH OCR START")
    print(f"Workspace root: {ROOT}")
    print(f"Claims dir: {CLAIMS_DIR}")
    print(f"Total PDF files found: {len(pdf_files)}")
    print("=" * 72)

    for idx, pdf_path in enumerate(pdf_files, start=1):
        try:
            # Backward-compatible call handling for older/newer function signatures.
            try:
                result = extract_text_from_pdf(pdf_path, max_pages=MAX_PAGES, use_gpu=False, debug=False)
            except TypeError:
                result = extract_text_from_pdf(pdf_path, max_pages=MAX_PAGES, use_gpu=False)

            if isinstance(result, tuple) and len(result) == 3:
                text, method, err = result
            elif isinstance(result, tuple) and len(result) == 2:
                text, method = result
                err = ""
            else:
                raise RuntimeError("extract_text_from_pdf returned unexpected result format")

            char_count = len(text)
            low_quality = char_count < LOW_QUALITY_THRESHOLD

            # Stable unique filename to avoid collisions across folders.
            short_hash = hashlib.md5(str(pdf_path).encode("utf-8")).hexdigest()[:10]
            txt_name = f"{pdf_path.stem}_{short_hash}.txt"
            txt_path = ocr_text_dir / txt_name
            txt_path.write_text(text, encoding="utf-8", errors="ignore")

            records.append({
                "file_path": str(pdf_path),
                "ocr_method": method,
                "char_count": char_count,
                "low_quality": low_quality,
                "text_file": str(txt_path),
                "error": err,
            })

            print(f"[{idx:04d}/{len(pdf_files)}] OK | chars={char_count:5d} | method={method} | {pdf_path.name}")
        except Exception as e:
            records.append({
                "file_path": str(pdf_path),
                "ocr_method": "error",
                "char_count": 0,
                "low_quality": True,
                "text_file": "",
                "error": repr(e),
            })
            print(f"[{idx:04d}/{len(pdf_files)}] ERROR | {pdf_path.name} | {repr(e)}")

    csv_path = OUT_DIR / "ocr_batch_results.csv"
    fieldnames = ["file_path", "ocr_method", "char_count", "low_quality", "text_file", "error"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in records:
            writer.writerow({k: row.get(k, "") for k in fieldnames})

    num_low = sum(1 for r in records if r.get("low_quality", True))
    num_error = sum(1 for r in records if r.get("ocr_method") == "error")
    num_nonempty = sum(1 for r in records if r.get("char_count", 0) > 0)

    print("\n" + "=" * 72)
    print("FULL BATCH OCR COMPLETE")
    print("Processed:", len(records))
    print("Non-empty rows:", num_nonempty)
    print("Low-quality rows:", num_low)
    print("Error rows:", num_error)
    print("CSV saved:", csv_path)
    print("Text outputs dir:", ocr_text_dir)

Batch OCR is skipped (RUN_BATCH_OCR=False).
Set RUN_BATCH_OCR=True only if you want to regenerate OCR outputs.


## Cell 7: Image analysis over full Claims batch
What this does:
- Scans all image files under Claims recursively.
- Computes quality features useful for triage and fraud-risk heuristics.
- Exports a CSV summary for downstream use.

Output file:
- outputs/image_analysis_results.csv

In [1]:
# Cell 7 code: Analyze all images and export CSV
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Self-healing setup so this cell can run independently.
if "ROOT" not in globals():
    ROOT = Path.cwd()
if "CLAIMS_DIR" not in globals():
    CLAIMS_DIR = ROOT / "dataset" / "Claims"
if "OUT_DIR" not in globals():
    OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

img_files = sorted([
    p for p in CLAIMS_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

if not img_files:
    raise RuntimeError(f"No image files found under: {CLAIMS_DIR}")

def _safe_rel(path_obj, base):
    try:
        return str(path_obj.relative_to(base))
    except Exception:
        return str(path_obj)

def _edge_density(gray_float):
    # Simple gradient-based edge proxy in [0,1] range.
    gx = np.abs(np.diff(gray_float, axis=1))
    gy = np.abs(np.diff(gray_float, axis=0))
    h = min(gx.shape[0], gy.shape[0])
    w = min(gx.shape[1], gy.shape[1])
    if h == 0 or w == 0:
        return 0.0
    g = gx[:h, :w] + gy[:h, :w]
    return float((g > 25.0).mean())

records = []
print("=" * 72)
print("IMAGE ANALYSIS START")
print(f"Claims dir: {CLAIMS_DIR}")
print(f"Total image files found: {len(img_files)}")
print("=" * 72)

for idx, img_path in enumerate(img_files, start=1):
    try:
        with Image.open(img_path) as im:
            rgb = im.convert("RGB")
            arr = np.asarray(rgb, dtype=np.uint8)

        gray = arr.mean(axis=2).astype(np.float32)
        h, w = gray.shape
        px = int(h * w)
        mean_brightness = float(gray.mean())
        std_brightness = float(gray.std())
        edge_density = _edge_density(gray)

        # Practical flags for weak quality images.
        low_light = mean_brightness < 55.0
        low_contrast = std_brightness < 28.0
        tiny_image = px < (600 * 600)

        records.append({
            "file_path": str(img_path),
            "relative_path": _safe_rel(img_path, CLAIMS_DIR),
            "file_name": img_path.name,
            "suffix": img_path.suffix.lower(),
            "width": int(w),
            "height": int(h),
            "pixels": px,
            "megapixels": round(px / 1_000_000.0, 4),
            "mean_brightness": round(mean_brightness, 3),
            "std_brightness": round(std_brightness, 3),
            "edge_density": round(edge_density, 5),
            "low_light": bool(low_light),
            "low_contrast": bool(low_contrast),
            "tiny_image": bool(tiny_image),
            "file_size_kb": round(img_path.stat().st_size / 1024.0, 3),
            "error": "",
        })
        print(f"[{idx:04d}/{len(img_files)}] OK | {img_path.name}")
    except Exception as e:
        records.append({
            "file_path": str(img_path),
            "relative_path": _safe_rel(img_path, CLAIMS_DIR),
            "file_name": img_path.name,
            "suffix": img_path.suffix.lower(),
            "width": 0,
            "height": 0,
            "pixels": 0,
            "megapixels": 0.0,
            "mean_brightness": 0.0,
            "std_brightness": 0.0,
            "edge_density": 0.0,
            "low_light": True,
            "low_contrast": True,
            "tiny_image": True,
            "file_size_kb": 0.0,
            "error": repr(e),
        })
        print(f"[{idx:04d}/{len(img_files)}] ERROR | {img_path.name} | {repr(e)}")

df_img = pd.DataFrame(records)
csv_path = OUT_DIR / "image_analysis_results.csv"
df_img.to_csv(csv_path, index=False, encoding="utf-8")

print("\n" + "=" * 72)
print("IMAGE ANALYSIS COMPLETE")
print("Processed:", len(df_img))
print("Errors:", int((df_img["error"] != "").sum()))
print("Low-light:", int(df_img["low_light"].sum()))
print("Low-contrast:", int(df_img["low_contrast"].sum()))
print("Tiny-image:", int(df_img["tiny_image"].sum()))
print("CSV saved:", csv_path)

IMAGE ANALYSIS START
Claims dir: c:\AB-PMJAY-Hackathon\dataset\Claims
Total image files found: 37
[0001/37] OK | 000012__BOCW_GJ_R3_2026040310046613__4d079c0c-a8a0-417e-87aa-6f277f84fa5a.jpg
[0002/37] OK | 000032__BOCW_GJ_R3_2026040310046613__1b2c9239-9dfc-4dd3-9d47-7e4ba5e6e498.jpg
[0003/37] OK | 000022__GOV_GJ_R2_2025_2026040410000059__PTCA_IMAGE.jpeg
[0004/37] OK | 000024__GOV_GJ_R2_2025_2026040410000059__PTCA_REPORT_001.jpg
[0005/37] OK | 000032__GOV_GJ_R2_2025_2026040410000059__CAG_REPORT_001.jpg
[0006/37] OK | 000034__GOV_GJ_R2_2025_2026040410000059__CAG_IMAGE.jpeg
[0007/37] OK | 000035__MAV_GJ_R3_2026040310038930__POST_CAG_DAIGARM.jpg
[0008/37] OK | 000036__MAV_GJ_R3_2026040310038930__POST_ANGI_REPORT.jpg
[0009/37] OK | 000038__MAV_GJ_R3_2026040310038930__POST_STENT.jpeg
[0010/37] OK | 000046__MAV_GJ_R3_2026040310038930__CAG_DAIGARM.jpg
[0011/37] OK | 000072__MAV_GJ_R3_2026040310038930__CAG_01.jpeg
[0012/37] OK | 000074__MAV_GJ_R3_2026040310038930__CAG_02.jpeg
[0013/37] OK | 000

## Cell 8: Generate claim-level aggregated table
What this does:
- Reads outputs/ocr_batch_results.csv and outputs/image_analysis_results.csv.
- Extracts claim identifiers from file paths.
- Aggregates file-level signals into one row per claim.

Output file:
- outputs/claims_aggregated.csv

In [5]:
# Cell 8 code: Build claim-level aggregated CSV
from pathlib import Path
import pandas as pd

if "ROOT" not in globals():
    ROOT = Path.cwd()
if "OUT_DIR" not in globals():
    OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ocr_csv = OUT_DIR / "ocr_batch_results.csv"
img_csv = OUT_DIR / "image_analysis_results.csv"

has_ocr = ocr_csv.exists()
has_img = img_csv.exists()

if not has_ocr and not has_img:
    raise RuntimeError(
        f"Neither input exists. Expected at least one of: {ocr_csv} or {img_csv}"
    )

if has_ocr:
    df_ocr = pd.read_csv(ocr_csv)
else:
    print(f"WARNING: OCR results not found, proceeding without OCR: {ocr_csv}")
    df_ocr = pd.DataFrame(
        columns=["file_path", "ocr_method", "char_count", "low_quality", "text_file", "error"]
    )

if has_img:
    df_img = pd.read_csv(img_csv)
else:
    print(f"WARNING: Image analysis results not found, proceeding without image data: {img_csv}")
    df_img = pd.DataFrame(
        columns=[
            "file_path",
            "low_light",
            "low_contrast",
            "tiny_image",
            "error",
        ]
    )

def _safe_bool_series(df, col):
    if col not in df.columns:
        return pd.Series([False] * len(df), index=df.index)
    s = df[col]
    if s.dtype == bool:
        return s.fillna(False)
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes", "y"])

def _extract_claim_keys(path_str):
    parts = Path(str(path_str)).parts
    if "Claims" in parts:
        i = parts.index("Claims")
        pkg = parts[i + 1] if len(parts) > i + 1 else "UNKNOWN_PKG"
        claim = parts[i + 2] if len(parts) > i + 2 else "UNKNOWN_CLAIM"
        return pkg, claim
    return "UNKNOWN_PKG", "UNKNOWN_CLAIM"

def _add_claim_keys(df):
    if "file_path" not in df.columns:
        df["file_path"] = ""
    key_rows = [_extract_claim_keys(p) for p in df["file_path"].tolist()]
    keys_df = pd.DataFrame(key_rows, columns=["package_code", "claim_id"], index=df.index)
    df[["package_code", "claim_id"]] = keys_df
    return df

# -------- OCR claim-level metrics --------
df_ocr = _add_claim_keys(df_ocr)

if "char_count" not in df_ocr.columns:
    df_ocr["char_count"] = 0
if "ocr_method" not in df_ocr.columns:
    df_ocr["ocr_method"] = "unknown"
if "error" not in df_ocr.columns:
    df_ocr["error"] = ""
if "low_quality" not in df_ocr.columns:
    df_ocr["low_quality"] = False

df_ocr["char_count"] = pd.to_numeric(df_ocr["char_count"], errors="coerce").fillna(0).astype(int)
df_ocr["low_quality_bool"] = _safe_bool_series(df_ocr, "low_quality")
df_ocr["ocr_error_bool"] = df_ocr["error"].fillna("").astype(str).str.strip().ne("") | df_ocr["ocr_method"].fillna("").astype(str).eq("error")
df_ocr["ocr_nonempty_bool"] = df_ocr["char_count"] > 0

agg_ocr = (
    df_ocr.groupby(["package_code", "claim_id"], dropna=False)
    .agg(
        ocr_pdf_docs=("file_path", "count"),
        ocr_nonempty_docs=("ocr_nonempty_bool", "sum"),
        ocr_low_quality_docs=("low_quality_bool", "sum"),
        ocr_error_docs=("ocr_error_bool", "sum"),
        ocr_total_chars=("char_count", "sum"),
        ocr_avg_chars=("char_count", "mean"),
        ocr_max_chars=("char_count", "max"),
    )
    .reset_index()
)
if len(agg_ocr) > 0:
    agg_ocr["ocr_avg_chars"] = agg_ocr["ocr_avg_chars"].round(2)

# -------- Image claim-level metrics --------
df_img = _add_claim_keys(df_img)

for col in ["low_light", "low_contrast", "tiny_image"]:
    if col not in df_img.columns:
        df_img[col] = False
if "error" not in df_img.columns:
    df_img["error"] = ""

df_img["low_light_bool"] = _safe_bool_series(df_img, "low_light")
df_img["low_contrast_bool"] = _safe_bool_series(df_img, "low_contrast")
df_img["tiny_image_bool"] = _safe_bool_series(df_img, "tiny_image")
df_img["img_error_bool"] = df_img["error"].fillna("").astype(str).str.strip().ne("")

agg_img = (
    df_img.groupby(["package_code", "claim_id"], dropna=False)
    .agg(
        image_docs=("file_path", "count"),
        image_low_light_docs=("low_light_bool", "sum"),
        image_low_contrast_docs=("low_contrast_bool", "sum"),
        image_tiny_docs=("tiny_image_bool", "sum"),
        image_error_docs=("img_error_bool", "sum"),
    )
    .reset_index()
)

# -------- Final merge --------
if len(agg_ocr) == 0 and len(agg_img) == 0:
    claims_agg = pd.DataFrame(
        columns=[
            "package_code", "claim_id", "ocr_pdf_docs", "ocr_nonempty_docs",
            "ocr_low_quality_docs", "ocr_error_docs", "ocr_total_chars",
            "ocr_avg_chars", "ocr_max_chars", "image_docs", "image_low_light_docs",
            "image_low_contrast_docs", "image_tiny_docs", "image_error_docs", "doc_total",
        ]
    )
else:
    claims_agg = pd.merge(
        agg_ocr,
        agg_img,
        on=["package_code", "claim_id"],
        how="outer",
    )

    num_cols = [
        c for c in claims_agg.columns
        if c not in ["package_code", "claim_id"]
    ]
    claims_agg[num_cols] = claims_agg[num_cols].fillna(0)

    for col in num_cols:
        if col != "ocr_avg_chars":
            claims_agg[col] = claims_agg[col].astype(int)

    claims_agg["doc_total"] = claims_agg.get("ocr_pdf_docs", 0) + claims_agg.get("image_docs", 0)

claims_agg = claims_agg.sort_values(["package_code", "claim_id"]).reset_index(drop=True)

out_csv = OUT_DIR / "claims_aggregated.csv"
claims_agg.to_csv(out_csv, index=False, encoding="utf-8")

print("=" * 72)
print("CLAIM AGGREGATION COMPLETE")
print("Claims rows:", len(claims_agg))
print("OCR source used:", has_ocr)
print("Image source used:", has_img)
if "ocr_pdf_docs" in claims_agg.columns:
    print("Total OCR docs:", int(claims_agg["ocr_pdf_docs"].sum()))
if "image_docs" in claims_agg.columns:
    print("Total image docs:", int(claims_agg["image_docs"].sum()))
print("CSV saved:", out_csv)
print("Preview:")
print(claims_agg.head(10).to_string(index=False))

CLAIM AGGREGATION COMPLETE
Claims rows: 14
OCR source used: False
Image source used: True
Total OCR docs: 0
Total image docs: 37
CSV saved: c:\AB-PMJAY-Hackathon\outputs\claims_aggregated.csv
Preview:
package_code                              claim_id  ocr_pdf_docs  ocr_nonempty_docs  ocr_low_quality_docs  ocr_error_docs  ocr_total_chars  ocr_avg_chars  ocr_max_chars  image_docs  image_low_light_docs  image_low_contrast_docs  image_tiny_docs  image_error_docs  doc_total
      MC011A        BOCW_GJ_R3_2026040310046613_ER             0                  0                     0               0                0            0.0              0           2                     0                        0                0                 0          2
      MC011A       GOV_GJ_R2_2025_2026040410000059             0                  0                     0               0                0            0.0              0           4                     0                        0                0       

## Cell 9: NHA vision model inference
What this does:
- Loads NHA credentials safely from environment variables or local client credentials file.
- Encodes claim images to data URL and sends them to NHA completion API.
- Runs single-image test first, then optional small batch.
- Saves outputs/nha_image_inference.csv for downstream review.

Before running:
- Put your model name in NHA_MODEL.
- Keep batch size small first to control API usage and cost.

In [ ]:
# Cell 9 code: NHA vision API integration (single + batch)
import os
import json
import base64
from pathlib import Path
import pandas as pd

try:
    from nha_client import NHAclient
except Exception as e:
    raise RuntimeError(
        "nha_client package is not available in this environment. Install it in the same notebook kernel first. "
        f"Original error: {repr(e)}"
    )

if "ROOT" not in globals():
    ROOT = Path.cwd()
if "CLAIMS_DIR" not in globals():
    CLAIMS_DIR = ROOT / "dataset" / "Claims"
if "OUT_DIR" not in globals():
    OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# 1) Credentials loading
# -----------------------
def _load_nha_credentials(root_dir: Path):
    env_id = os.getenv("NHA_CLIENT_ID", "").strip()
    env_secret = os.getenv("NHA_CLIENT_SECRET", "").strip()
    if env_id and env_secret:
        return env_id, env_secret, "environment"

    cred_path = root_dir / "client-credentials.json"
    if cred_path.exists():
        with open(cred_path, "r", encoding="utf-8") as f:
            cred = json.load(f)
        file_id = str(cred.get("clientId", "")).strip()
        file_secret = str(cred.get("clientSecret", "")).strip()
        if file_id and file_secret:
            return file_id, file_secret, str(cred_path)

    raise RuntimeError(
        "NHA credentials not found. Set NHA_CLIENT_ID and NHA_CLIENT_SECRET environment variables "
        "or create client-credentials.json with clientId and clientSecret keys."
    )

client_id, client_secret, cred_source = _load_nha_credentials(ROOT)
print("Credentials source:", cred_source)
print("Client ID loaded:", "yes" if client_id else "no")
print("Client Secret loaded:", "yes" if client_secret else "no")

# -----------------------
# 2) Model + prompt setup
# -----------------------
NHA_MODEL = ""  # TODO: set the exact model name shared by NHA
PROMPT_TEXT = "Describe key clinical findings visible in this image in 3-5 bullet points."
BATCH_LIMIT = 10  # keep small initially to control API calls
RUN_BATCH = False  # first run single-image test, then set True if output looks good

if not NHA_MODEL.strip():
    raise RuntimeError("Please set NHA_MODEL before running this cell.")

nc = NHAclient(client_id, client_secret)

def _image_to_data_url(image_path: Path) -> str:
    suffix = image_path.suffix.lower()
    mime = {
        ".jpg": "image/jpeg",
.jpeg": "image/jpeg",
.png": "image/png",
.bmp": "image/bmp",
.tif": "image/tiff",
.tiff": "image/tiff",
.webp": "image/webp",
    }.get(suffix, "image/jpeg")

    with open(image_path, "rb") as f:
        image_bytes = f.read()
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:{mime};base64,{image_base64}"

def _extract_text_from_response(resp):
    try:
        if isinstance(resp, dict):
            choices = resp.get("choices")
            if isinstance(choices, list) and choices:
                msg = choices[0].get("message", {})
                content = msg.get("content", "")
                if isinstance(content, str):
                    return content
                if isinstance(content, list):
                    parts = []
                    for item in content:
                        if isinstance(item, dict) and item.get("type") == "text":
                            parts.append(str(item.get("text", "")))
                    return "\n".join([p for p in parts if p]).strip()
            return json.dumps(resp, ensure_ascii=False)[:4000]
        return str(resp)[:4000]
    except Exception:
        return str(resp)[:4000]

def run_nha_vision(image_path: Path, prompt: str, model_name: str):
    data_url = _image_to_data_url(image_path)
    response = nc.completion(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": data_url}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
        metadata={"problem_statement": 2},
    )
    return response

# -----------------------
# 3) Single-image smoke test
# -----------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
img_files = sorted([
    p for p in CLAIMS_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

if not img_files:
    raise RuntimeError(f"No images found under: {CLAIMS_DIR}")

test_image = img_files[0]
print("Single test image:", test_image)

single_response = run_nha_vision(test_image, PROMPT_TEXT, NHA_MODEL)
single_text = _extract_text_from_response(single_response)
print("\nSingle-image model output preview:")
print(single_text[:1200])

# -----------------------
# 4) Optional batch run
# -----------------------
batch_records = []
if RUN_BATCH:
    selected = img_files[: max(1, BATCH_LIMIT)]
    print(f"\nBatch mode ON. Processing {len(selected)} images...")

    for idx, img_path in enumerate(selected, start=1):
        try:
            resp = run_nha_vision(img_path, PROMPT_TEXT, NHA_MODEL)
            out_text = _extract_text_from_response(resp)
            batch_records.append(
                {
                    "file_path": str(img_path),
                    "file_name": img_path.name,
                    "model": NHA_MODEL,
                    "prompt": PROMPT_TEXT,
                    "response_text": out_text,
                    "error": "",
                }
            )
            print(f"[{idx:03d}/{len(selected)}] OK | {img_path.name}")
        except Exception as e:
            batch_records.append(
                {
                    "file_path": str(img_path),
                    "file_name": img_path.name,
                    "model": NHA_MODEL,
                    "prompt": PROMPT_TEXT,
                    "response_text": "",
                    "error": repr(e),
                }
            )
            print(f"[{idx:03d}/{len(selected)}] ERROR | {img_path.name} | {repr(e)}")

    df_nha = pd.DataFrame(batch_records)
    nha_out = OUT_DIR / "nha_image_inference.csv"
    df_nha.to_csv(nha_out, index=False, encoding="utf-8")
    print("\nSaved:", nha_out)
    print("Rows:", len(df_nha))
else:
    print("\nBatch mode OFF. Set RUN_BATCH=True to create outputs/nha_image_inference.csv.")